In [ ]:
# Cluster task. Connects to Lakebase with a Databricks-minted OAuth token (no stored password).
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)
import state_store as ss
from state_store import StateStore


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
# Apply lakebase_schema.sql (control, checkpoints, reseed queue, telemetry) in ONE transaction:
# the pending renames (state_store.RENAMES: ct_checkpoint -> checkpoint, sweep_run -> ingest_run, ...),
# every DDL statement, then the schema version (state_store.SCHEMA_VERSION). Re-applying is a no-op,
# and a failure leaves the database unchanged. A load holding a table makes the apply wait and retry;
# if it gives up, this cell fails with a message to run it again. Ingestion runs apply a pending migration the same way
# (StateStore.migrate), so running this after an upgrade is optional.
schema_path = "/Workspace" + nb_dir + "/lakebase_schema.sql"
with open(schema_path) as f:
    ddl = f.read()
before = store.schema_version()
result = store.migrate(ddl, force=True)
for stmt in result["renames"]:
    print(stmt)
print(f"applied {len(result['renames'])} renames and {result['ran']} DDL statements "
      f"({result['skipped']} already in place) to {db_name}: schema version {before} -> {ss.SCHEMA_VERSION}")

In [ ]:
# Describe every table and column (lakebase_comments.sql), as Postgres comments: a Lakebase database registered in
# Unity Catalog shows them as the tables' and columns' comments, so Genie can answer questions about runs and
# freshness. Applied on every run (they carry no schema version). Unity Catalog refreshes a Lakebase table's
# comments when a query resolves it, so each table is then described through the catalog (lakebase_catalog), which
# also lists it there. A failure only warns: descriptions never fail setup.
import re
lakebase_catalog = widget("lakebase_catalog", "").strip()
try:
    with open("/Workspace" + nb_dir + "/lakebase_comments.sql") as f:
        comments = f.read()
    print(f"applied {store.apply_comments(comments)} table and column descriptions")
    if lakebase_catalog:
        described = 0
        for table in sorted(set(re.findall(r"comment on table lakefed_ingest\.(\w+)", comments))):
            try:
                spark.sql(f"DESCRIBE TABLE EXTENDED `{lakebase_catalog}`.lakefed_ingest.`{table}`").collect()
                described += 1
            except Exception as e:
                print(f"[warn] could not describe {lakebase_catalog}.lakefed_ingest.{table}: {str(e)[:200]}")
        print(f"described {described} tables through {lakebase_catalog}, so Unity Catalog shows their descriptions")
except Exception as e:
    print(f"[warn] could not apply the table and column descriptions; setup goes on: {str(e)[:300]}")
